# 🎯 STAGE 3: QWEN Knowledge Distillation to Smaller Sizes

## Paper Objective: Finding the Pareto Frontier - Maximum Compression vs Arabic Legal Performance

**Teacher Model:** FANAR-9B + SFT (Stage 1 best: 77.60% accuracy without context)

**Student Models (4 sizes):**
1. **7B**: Qwen2.5-7B-Instruct
2. **3B**: Qwen2.5-3B-Instruct
3. **1.5B**: Qwen2.5-1.5B-Instruct
4. **500M**: Qwen2.5-0.5B-Instruct

**Method:** Task-specific Knowledge Distillation
- Loss: α·CE(student, true) + (1-α)·KL(student_logits, teacher_logits)
- α = 0.3 (emphasizes teacher knowledge)
- Temperature T = 4.0
- LoRA fine-tuning for efficiency

**Then:** Apply Stage 2 best quantization (INT4 NF4) to each distilled model

**Final Output:** 8 new models + Pareto frontier with Stages 1 & 2

## 📦 CELL 1: Install Dependencies

In [45]:
# Install required packages
!pip install -q torch transformers accelerate bitsandbytes
!pip install -q peft datasets trl
!pip install -q sentencepiece protobuf
!pip install -q rouge-score nltk
!pip install -q matplotlib seaborn pandas tqdm

print("✅ Dependencies installed")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
✅ Dependencies installed


## 📚 CELL 2: Imports and Configuration

In [46]:
!pip install "transformers==4.47.1"

Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-4.47.1-py3-none-any.whl.metadata (44 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.47.1-py3-none-any.whl (10.1 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:━━━━━━━━ 0/3 [huggingface-hub]
      Successfully uninstalled huggingface_hub-1.19.0m0/3 [huggingface-hub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [huggingface-hub]  WARNING: The scripts hf, huggingface-cli and tiny-agents are installed in '/home/pmp6991/.local/bin' which is not on PATH.
  Consider adding thi

In [1]:
import torch
import transformers
import bitsandbytes
import peft

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)

/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.12.1+cu130
CUDA: True
Transformers: 5.12.1


In [6]:
import os
import json
import re
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
from datasets import Dataset as HFDataset
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

class Config:
    """Stage 3 Configuration - extends Stage 1 & 2"""
    
    # Teacher model (from Stage 1 SFT)
    TEACHER_MODEL_BASE = "Qwen/Qwen2.5-7B-Instruct"
    TEACHER_LORA_PATH = "./models_arablegal/exp_c_sft"  # Your Stage 1 SFT model
    
    # Student base models (4 sizes)
    STUDENT_MODELS = {
        '7B':   'Qwen/Qwen2.5-7B-Instruct',
        '3B':   'Qwen/Qwen2.5-3B-Instruct',
        '1.5B': 'Qwen/Qwen2.5-1.5B-Instruct',
        '500M': 'Qwen/Qwen2.5-0.5B-Instruct',
    }
    
    # Output directories
    OUTPUT_DIR = "./outputs_QWEN_stage3"
    RESULTS_DIR = "./results_QWEN_stage3"
    MODELS_DIR = "./models_QWEN_stage3"
    DATA_DIR = "./results_arablegal"  # Where Stage 1 splits are
    
    # Distillation hyperparameters
    ALPHA = 0.3  # Weight for CE loss (1-α for KL distillation loss)
    TEMPERATURE = 4.0  # Distillation temperature
    
    # Training
    NUM_EPOCHS = 3
    BATCH_SIZE = 2  # Smaller for distillation (memory)
    GRADIENT_ACCUMULATION = 8  # Effective batch = 16
    LEARNING_RATE = 2e-4
    WARMUP_RATIO = 0.1
    MAX_LENGTH = 1024
    
    # LoRA for students
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"]
    
    # Quantization
    USE_4BIT_TEACHER = True  # Load teacher in 4-bit for memory
    
    # Evaluation (same as Stage 1 & 2)
    EVAL_SAMPLE_SIZE = 500
    MAX_NEW_TOKENS = 10
    TEMPERATURE_GEN = 0.1
    TOP_P = 0.9
    
    SEED = 42

# Create directories
for d in [Config.OUTPUT_DIR, Config.RESULTS_DIR, Config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Configuration set")
print(f"   Teacher: {Config.TEACHER_LORA_PATH}")
print(f"   Student sizes: {list(Config.STUDENT_MODELS.keys())}")
print(f"   Distillation α: {Config.ALPHA}, Temperature: {Config.TEMPERATURE}")

# GPU check
if torch.cuda.is_available():
    print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ Configuration set
   Teacher: ./models_arablegal/exp_c_sft
   Student sizes: ['7B', '3B', '1.5B', '500M']
   Distillation α: 0.3, Temperature: 4.0

✅ GPU: NVIDIA A100-PCIE-40GB
   Memory: 42.43 GB


In [7]:
# ──────────────────────────────────────────────────────────────────────
# COMPREHENSIVE PATCH for QWEN models on transformers 5.x
# fa uses GPT-2 style Conv1D layers that require several utility 
# functions removed in transformers 5.x. We re-inject the 4.x implementations.
# ──────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import transformers.pytorch_utils as _pu

print("🔧 Patching transformers.pytorch_utils for Jais compatibility...")
print(f"   Transformers version: {__import__('transformers').__version__}\n")

# ──────────────────────────────────────────────────────────────────────
# Patch 1: find_pruneable_heads_and_indices
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "find_pruneable_heads_and_indices"):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _pu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
    print("   ✅ Patched: find_pruneable_heads_and_indices")
else:
    print("   ℹ️  find_pruneable_heads_and_indices already present")

# ──────────────────────────────────────────────────────────────────────
# Patch 2: Conv1D class (GPT-2 style)
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "Conv1D"):
    class Conv1D(nn.Module):
        """1D-convolutional layer as used in GPT-2 / Jais.
        Equivalent to a Linear but with weight shape (nx, nf) instead of (nf, nx).
        """
        def __init__(self, nf, nx):
            super().__init__()
            self.nf = nf
            self.weight = nn.Parameter(torch.empty(nx, nf))
            self.bias = nn.Parameter(torch.zeros(nf))
            nn.init.normal_(self.weight, std=0.02)
        
        def forward(self, x):
            size_out = x.size()[:-1] + (self.nf,)
            x = torch.addmm(self.bias, x.view(-1, x.size(-1)), self.weight)
            x = x.view(*size_out)
            return x
    _pu.Conv1D = Conv1D
    print("   ✅ Patched: Conv1D")
else:
    print("   ℹ️  Conv1D already present")

# ──────────────────────────────────────────────────────────────────────
# Patch 3: prune_conv1d_layer
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "prune_conv1d_layer"):
    def prune_conv1d_layer(layer, index, dim=1):
        """Prune a Conv1D layer to keep only entries in index."""
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        if dim == 0:
            b = layer.bias.clone().detach()
        else:
            b = layer.bias[index].clone().detach()
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = _pu.Conv1D(new_size[1], new_size[0]).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        new_layer.bias.requires_grad = False
        new_layer.bias.copy_(b.contiguous())
        new_layer.bias.requires_grad = True
        return new_layer
    _pu.prune_conv1d_layer = prune_conv1d_layer
    print("   ✅ Patched: prune_conv1d_layer")
else:
    print("   ℹ️  prune_conv1d_layer already present")

# ──────────────────────────────────────────────────────────────────────
# Patch 4: prune_linear_layer (just in case)
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "prune_linear_layer"):
    def prune_linear_layer(layer, index, dim=0):
        """Prune a linear layer to keep only entries in index."""
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        if layer.bias is not None:
            if dim == 1:
                b = layer.bias.clone().detach()
            else:
                b = layer.bias[index].clone().detach()
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        if layer.bias is not None:
            new_layer.bias.requires_grad = False
            new_layer.bias.copy_(b.contiguous())
            new_layer.bias.requires_grad = True
        return new_layer
    _pu.prune_linear_layer = prune_linear_layer
    print("   ✅ Patched: prune_linear_layer")
else:
    print("   ℹ️  prune_linear_layer already present")

# ──────────────────────────────────────────────────────────────────────
# Patch 5: prune_layer (general pruning utility)
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "prune_layer"):
    def prune_layer(layer, index, dim=None):
        """Prune a layer - dispatches to Conv1D or Linear pruning."""
        if isinstance(layer, _pu.Conv1D):
            if dim is None:
                dim = 1
            return _pu.prune_conv1d_layer(layer, index, dim=dim)
        elif isinstance(layer, nn.Linear):
            if dim is None:
                dim = 0
            return _pu.prune_linear_layer(layer, index, dim=dim)
        else:
            raise ValueError(f"Can't prune layer of class {layer.__class__}")
    _pu.prune_layer = prune_layer
    print("   ✅ Patched: prune_layer")
else:
    print("   ℹ️  prune_layer already present")

print("\n✅ All transformers compatibility patches applied!")
print("   Ready to load QWEN models")

🔧 Patching transformers.pytorch_utils for Jais compatibility...
   Transformers version: 5.12.1

   ℹ️  find_pruneable_heads_and_indices already present
   ℹ️  Conv1D already present
   ℹ️  prune_conv1d_layer already present
   ℹ️  prune_linear_layer already present
   ℹ️  prune_layer already present

✅ All transformers compatibility patches applied!
   Ready to load QWEN models


In [37]:
import os
import json
import re
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
from datasets import Dataset as HFDataset
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

class Config:
    """Stage 3 Configuration - JAIS family distillation"""
    
    # Teacher (Jais-2-8B SFT from Stage 1)
    TEACHER_MODEL_BASE = "Qwen/Qwen2.5-7B-Instruct"
    TEACHER_LORA_PATH = "./models_QWENlegal_stage1/exp_c_sft"
    
    # Student base models - ALL Jais family for proper distillation
    # ⚠️ FIXED: jais-13b-chat is 13B (was incorrectly labeled 6B)
    STUDENT_MODELS = {
        '7B':   'Qwen/Qwen2.5-7B-Instruct',
        '3B':   'Qwen/Qwen2.5-3B-Instruct',
        '1.5B': 'Qwen/Qwen2.5-1.5B-Instruct',
        '500M': 'Qwen/Qwen2.5-0.5B-Instruct',
    }
    # Output directories
    OUTPUT_DIR = "./outputs_stage3_QWEN"
    RESULTS_DIR = "./results_qwen_stage3"
    MODELS_DIR = "./models_qwen_stage3"
    DATA_DIR = "./results_fanar_stage1"  # Where data splits are
    
    # Distillation hyperparameters
    ALPHA = 0.3
    TEMPERATURE = 4.0
    
    # Training
    NUM_EPOCHS = 3
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION = 8
    LEARNING_RATE = 2e-4
    WARMUP_RATIO = 0.1
    MAX_LENGTH = 1024
    
    # LoRA
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    # ⚠️ FIXED: Jais uses GPT-style Conv1D - different target modules!
    #LORA_TARGET_MODULES = ["c_attn", "c_proj", "c_fc"]  # Jais architecture
    
    # Quantization
    USE_4BIT_TEACHER = True
    
    # Evaluation
    EVAL_SAMPLE_SIZE = 500
    MAX_NEW_TOKENS = 10
    TEMPERATURE_GEN = 0.1
    TOP_P = 0.9
    
    SEED = 42

# Create directories
for d in [Config.OUTPUT_DIR, Config.RESULTS_DIR, Config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Configuration set")
print(f"   Teacher: {Config.TEACHER_MODEL_BASE}")
print(f"   Teacher LoRA: {Config.TEACHER_LORA_PATH}")
print(f"   Student sizes: {list(Config.STUDENT_MODELS.keys())}")
print(f"   LoRA target modules: {Config.LORA_TARGET_MODULES}")

if torch.cuda.is_available():
    print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ Configuration set
   Teacher: Qwen/Qwen2.5-7B-Instruct
   Teacher LoRA: ./models_QWENlegal_stage1/exp_c_sft
   Student sizes: ['7B', '3B', '1.5B', '500M']
   LoRA target modules: ['c_attn', 'c_proj', 'c_fc']

✅ GPU: NVIDIA A100-PCIE-40GB
   Memory: 42.43 GB


In [38]:
# ──────────────────────────────────────────────────────────────────────
# FIX: transformers 5.x removed `find_pruneable_heads_and_indices` from
# transformers.pytorch_utils, but JAIS remote code (trust_remote_code=True)
# still imports it. Re-inject the 4.x implementation BEFORE loading any
# JAIS model. Safe no-op on transformers 4.x where the symbol still exists.
# ──────────────────────────────────────────────────────────────────────
import torch
import transformers.pytorch_utils as _pu

if not hasattr(_pu, "find_pruneable_heads_and_indices"):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _pu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
    print("✅ Patched find_pruneable_heads_and_indices into transformers.pytorch_utils")
else:
    print("ℹ️  find_pruneable_heads_and_indices already present — no patch needed")


ℹ️  find_pruneable_heads_and_indices already present — no patch needed


## 📂 CELL 3: Load Data from Stage 1

In [31]:
# Load same data splits used in Stages 1 & 2 for consistency
with open(f"{Config.DATA_DIR}/mcq_train.json", 'r', encoding='utf-8') as f:
    mcq_train = json.load(f)

with open(f"{Config.DATA_DIR}/mcq_eval.json", 'r', encoding='utf-8') as f:
    mcq_eval = json.load(f)

with open(f"{Config.DATA_DIR}/qa_train.json", 'r', encoding='utf-8') as f:
    qa_train = json.load(f)

with open(f"{Config.DATA_DIR}/qa_eval.json", 'r', encoding='utf-8') as f:
    qa_eval = json.load(f)

# Use same 500 sample as Stages 1 & 2 for direct comparison
mcq_eval_sample = mcq_eval[:Config.EVAL_SAMPLE_SIZE]

print(f"✅ Data loaded:")
print(f"   MCQ Train: {len(mcq_train)} | Eval: {len(mcq_eval)} (using {len(mcq_eval_sample)})")
print(f"   QA Train:  {len(qa_train)} | Eval: {len(qa_eval)}")
print(f"\n   Total training samples: {len(mcq_train) + len(qa_train)}")

✅ Data loaded:
   MCQ Train: 8451 | Eval: 2113 (using 500)
   QA Train:  1800 | Eval: 200

   Total training samples: 10251


In [32]:
import transformers, sys
print(transformers.__version__)
print(sys.executable)

5.12.1
/miniconda/bin/python


## 🔧 CELL 4: Helper Functions (Evaluation - Same as Stage 2)

In [33]:
def format_mcq_with_context(item):
    """With context - reading comprehension"""
    options = item['options']
    return f"""أنت خبير في القانون السعودي. أجب عن السؤال التالي بناءً على النص القانوني المرجعي.

النص القانوني المرجعي: {item.get('context', '')}

السؤال: {item['question']}

الخيارات:
A) {options['A']}
B) {options['B']}
C) {options['C']}
D) {options['D']}

اختر الإجابة الصحيحة (A أو B أو C أو D فقط):
الإجابة:"""

def format_mcq_without_context(item):
    """Without context - real knowledge test"""
    options = item['options']
    return f"""أنت خبير في القانون السعودي. أجب عن السؤال التالي بناءً على معرفتك بالأنظمة السعودية.

السؤال: {item['question']}

الخيارات:
A) {options['A']}
B) {options['B']}
C) {options['C']}
D) {options['D']}

اختر الإجابة الصحيحة (A أو B أو C أو D فقط):
الإجابة:"""

def extract_mcq_answer(response):
    """Extract A, B, C, D from response"""
    response = response.strip().upper()
    for char in response[:50]:
        if char in ['A', 'B', 'C', 'D']:
            return char
    ar_to_en = {'أ': 'A', 'ب': 'B', 'ج': 'C', 'د': 'D'}
    for ar, en in ar_to_en.items():
        if ar in response[:20]:
            return en
    return 'INVALID'

def generate_answer(model, tokenizer, prompt, max_new_tokens=10):
    """Generate answer from any model"""
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=1500
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=Config.TEMPERATURE_GEN,
            top_p=Config.TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

def evaluate_dual_setting(model, tokenizer, eval_data, model_name="model"):
    """Evaluate in BOTH settings (with/without context)"""
    print(f"\n📊 Evaluating: {model_name}")
    
    correct_with = 0
    start_with = time.time()
    for item in tqdm(eval_data, desc=f"{model_name} (with ctx)"):
        prompt = format_mcq_with_context(item)
        response = generate_answer(model, tokenizer, prompt)
        if extract_mcq_answer(response) == item['correct_answer']:
            correct_with += 1
    time_with = time.time() - start_with
    
    correct_without = 0
    start_without = time.time()
    for item in tqdm(eval_data, desc=f"{model_name} (no ctx)"):
        prompt = format_mcq_without_context(item)
        response = generate_answer(model, tokenizer, prompt)
        if extract_mcq_answer(response) == item['correct_answer']:
            correct_without += 1
    time_without = time.time() - start_without
    
    accuracy_with = correct_with / len(eval_data) * 100
    accuracy_without = correct_without / len(eval_data) * 100
    
    return {
        'model_name': model_name,
        'with_context': {
            'accuracy': accuracy_with,
            'correct': correct_with,
            'total': len(eval_data),
            'time_seconds': time_with
        },
        'without_context': {
            'accuracy': accuracy_without,
            'correct': correct_without,
            'total': len(eval_data),
            'time_seconds': time_without
        },
        'reading_skill': accuracy_with - accuracy_without
    }

def get_model_size_info(model):
    """Calculate model size"""
    total_params = sum(p.numel() for p in model.parameters())
    
    if hasattr(model, 'get_memory_footprint'):
        memory_mb = model.get_memory_footprint() / (1024 ** 2)
    else:
        memory_mb = 0
        for p in model.parameters():
            if p.dtype == torch.float32:
                memory_mb += p.numel() * 4
            elif p.dtype in [torch.float16, torch.bfloat16]:
                memory_mb += p.numel() * 2
            elif p.dtype == torch.int8:
                memory_mb += p.numel() * 1
            else:
                memory_mb += p.numel() * 0.5
        memory_mb = memory_mb / (1024 ** 2)
    
    return {
        'total_params': total_params,
        'total_params_billions': total_params / 1e9,
        'memory_mb': memory_mb,
        'memory_gb': memory_mb / 1024
    }

def cleanup_memory():
    """Free GPU memory"""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"   GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("✅ Helper functions defined")

✅ Helper functions defined


## 🏆 CELL 5: Load Jais-2-8B-Chat SFT Teacher Model

Load and merge the Stage 1 SFT model to use as our teacher for distillation.

In [ ]:
#from huggingface_hub import login

#login("")

In [35]:
import transformers
print(transformers.__version__)

5.12.1


In [39]:
print("="*70)
print("🏆 LOADING TEACHER MODEL: QWEN-2.5-7B-Instruct SFT")
print("="*70)

# Load teacher tokenizer
teacher_tokenizer = AutoTokenizer.from_pretrained(
    Config.TEACHER_MODEL_BASE,
    trust_remote_code=True,
    padding_side="left"
)
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

# Load teacher in 4-bit for memory efficiency during data generation
teacher_quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("📥 Loading QWEN-2.5-7B-Instruct in 4-bit...")
teacher_base = AutoModelForCausalLM.from_pretrained(
    Config.TEACHER_MODEL_BASE,
    quantization_config=teacher_quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print("📥 Loading SFT LoRA weights...")
teacher_model = PeftModel.from_pretrained(teacher_base, Config.TEACHER_LORA_PATH)
teacher_model.eval()

teacher_info = get_model_size_info(teacher_model)
print(f"\n✅ Teacher loaded successfully")
print(f"   Parameters: {teacher_info['total_params_billions']:.2f}B")
print(f"   Memory: {teacher_info['memory_gb']:.2f} GB")

🏆 LOADING TEACHER MODEL: QWEN-2.5-7B-Instruct SFT
📥 Loading QWEN-2.5-7B-Instruct in 4-bit...


Loading weights: 100%|██████████| 339/339 [00:07<00:00, 47.88it/s]


📥 Loading SFT LoRA weights...

✅ Teacher loaded successfully
   Parameters: 4.39B
   Memory: 5.22 GB


## 📝 CELL 6: Generate Teacher Distillation Data

For each training question, get the teacher's answer. This creates the supervised signal for students.

In [40]:
def generate_teacher_answer(item, is_mcq=True):
    """Get teacher's answer for a training item"""
    if is_mcq:
        # Use without-context format (real knowledge)
        prompt = format_mcq_without_context(item)
        response = generate_answer(teacher_model, teacher_tokenizer, prompt, max_new_tokens=10)
        teacher_answer = extract_mcq_answer(response)
        return teacher_answer if teacher_answer != 'INVALID' else item['correct_answer']
    else:
        # For QA - generate full answer
        prompt = f"""أنت خبير في القانون السعودي. أجب عن السؤال التالي بإجابة مباشرة ومختصرة.

السؤال: {item['question']}

الإجابة:"""
        response = generate_answer(teacher_model, teacher_tokenizer, prompt, max_new_tokens=200)
        return response.strip() or item.get('answer', '')

# Check if distillation data already exists
distill_data_path = f"{Config.RESULTS_DIR}/teacher_distillation_data.json"

if os.path.exists(distill_data_path):
    print(f"✅ Loading existing teacher distillation data...")
    with open(distill_data_path, 'r', encoding='utf-8') as f:
        distillation_data = json.load(f)
    print(f"   Loaded {len(distillation_data)} samples")
else:
    print("📝 Generating teacher predictions on training data...")
    print("   This is the supervised signal for student distillation")
    print("   Estimated time: 30-60 minutes\n")
    
    distillation_data = []
    
    # Process MCQ training data
    for item in tqdm(mcq_train, desc="Teacher MCQ"):
        teacher_answer = generate_teacher_answer(item, is_mcq=True)
        
        distillation_data.append({
            'type': 'MCQ',
            'question': item['question'],
            'context': item.get('context', ''),
            'options': item['options'],
            'true_answer': item['correct_answer'],
            'teacher_answer': teacher_answer,
        })
    
    # Process QA training data
    for item in tqdm(qa_train, desc="Teacher QA"):
        teacher_answer = generate_teacher_answer(item, is_mcq=False)
        
        distillation_data.append({
            'type': 'QA',
            'question': item['question'],
            'context': item.get('context', ''),
            'true_answer': item.get('answer', ''),
            'teacher_answer': teacher_answer,
        })
    
    # Save
    with open(distill_data_path, 'w', encoding='utf-8') as f:
        json.dump(distillation_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Generated {len(distillation_data)} distillation samples")
    print(f"   Saved to: {distill_data_path}")

# Quick analysis
mcq_samples = [d for d in distillation_data if d['type'] == 'MCQ']
qa_samples = [d for d in distillation_data if d['type'] == 'QA']

if mcq_samples:
    mcq_accuracy = sum(
        1 for d in mcq_samples 
        if d['teacher_answer'] == d['true_answer']
    ) / len(mcq_samples) * 100
    print(f"\n📊 Teacher accuracy on training MCQ: {mcq_accuracy:.2f}%")

print(f"\n📊 Distillation dataset:")
print(f"   MCQ samples: {len(mcq_samples)}")
print(f"   QA samples:  {len(qa_samples)}")

✅ Loading existing teacher distillation data...
   Loaded 10251 samples

📊 Teacher accuracy on training MCQ: 68.06%

📊 Distillation dataset:
   MCQ samples: 8451
   QA samples:  1800


In [41]:
import json
import os

# Check both files
files_to_check = [
    "./results_jais_stage3/teacher_distillation_data.json",
    "./results_jais_stage3/teacher_distillation_data_partial.json"
]

for filepath in files_to_check:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024*1024)
        
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        mcq = [d for d in data if d['type'] == 'MCQ']
        qa = [d for d in data if d['type'] == 'QA']
        
        print(f"📄 {filepath}")
        print(f"   Size: {size_mb:.2f} MB")
        print(f"   Total: {len(data)} samples")
        print(f"   MCQ: {len(mcq)}")
        print(f"   QA:  {len(qa)}")
        print()

📄 ./results_jais_stage3/teacher_distillation_data.json
   Size: 10.71 MB
   Total: 10251 samples
   MCQ: 8451
   QA:  1800

📄 ./results_jais_stage3/teacher_distillation_data_partial.json
   Size: 10.71 MB
   Total: 10251 samples
   MCQ: 8451
   QA:  1800



In [42]:
import json

with open("./results_jais_stage3/teacher_distillation_data.json", 'r', encoding='utf-8') as f:
    distillation_data = json.load(f)

print(f"✅ Loaded {len(distillation_data)} samples")
print(f"   MCQ: {sum(1 for d in distillation_data if d['type'] == 'MCQ')}")
print(f"   QA:  {sum(1 for d in distillation_data if d['type'] == 'QA')}")

✅ Loaded 10251 samples
   MCQ: 8451
   QA:  1800


In [43]:
# Verify functions exist
try:
    print("✅ prepare_distillation_dataset:", prepare_distillation_dataset)
    print("✅ distill_student:", distill_student)
except NameError as e:
    print(f"❌ Missing: {e}")
    print("   Run the function definition cells first!")

✅ prepare_distillation_dataset: <function prepare_distillation_dataset at 0x145d70d8b240>
✅ distill_student: <function distill_student at 0x145d70d8af20>


In [44]:
# Check if the variable exists and save what we have
try:
    print(f"Found {len(distillation_data)} samples in memory")
    
    # Save immediately!
    with open(f"{Config.RESULTS_DIR}/teacher_distillation_data_partial.json", 'w', encoding='utf-8') as f:
        json.dump(distillation_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Saved to: teacher_distillation_data_partial.json")
    
    mcq = [d for d in distillation_data if d['type'] == 'MCQ']
    qa = [d for d in distillation_data if d['type'] == 'QA']
    print(f"   MCQ: {len(mcq)}")
    print(f"   QA:  {len(qa)}")
except NameError:
    print("⚠️ distillation_data variable not found")
    print("   The MCQ progress is lost - you'll need to restart")

Found 10251 samples in memory
✅ Saved to: teacher_distillation_data_partial.json
   MCQ: 8451
   QA:  1800


In [45]:
import shutil

# Copy backup to the main filename that Cell 8 expects
shutil.copy(
    "./results_jais_stage3/teacher_distillation_data_partial.json",
    "./results_jais_stage3/teacher_distillation_data.json"
)

print("✅ Renamed backup to main file")

# Verify
import json
with open("./results_jais_stage3/teacher_distillation_data.json", 'r') as f:
    distillation_data = json.load(f)

mcq = [d for d in distillation_data if d['type'] == 'MCQ']
qa = [d for d in distillation_data if d['type'] == 'QA']

print(f"\n📊 Verified data:")
print(f"   Total: {len(distillation_data)}")
print(f"   MCQ: {len(mcq)}")
print(f"   QA:  {len(qa)}")

# Check teacher MCQ accuracy
if mcq:
    correct = sum(1 for d in mcq if d['teacher_answer'] == d['true_answer'])
    accuracy = correct / len(mcq) * 100
    print(f"\n📊 Teacher MCQ accuracy: {accuracy:.2f}%")

✅ Renamed backup to main file

📊 Verified data:
   Total: 10251
   MCQ: 8451
   QA:  1800

📊 Teacher MCQ accuracy: 68.06%


## 🧹 CELL 7: Free Teacher from GPU (Save for Phase 2)

In [46]:
# ══════════════════════════════════════════════════════════════════════
# COMPREHENSIVE JAIS COMPATIBILITY PATCH for transformers 5.x
# Run this BEFORE loading ANY Jais model
# ══════════════════════════════════════════════════════════════════════
import sys
import types
import math
import torch
import torch.nn as nn
import transformers
import transformers.pytorch_utils as _pu
import transformers.utils

print("🔧 Patching transformers for Jais compatibility...")
print(f"   Transformers version: {transformers.__version__}\n")

# ──────────────────────────────────────────────────────────────────────
# Patch 1: find_pruneable_heads_and_indices
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "find_pruneable_heads_and_indices"):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _pu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
    print("   ✅ Patched: find_pruneable_heads_and_indices")

# ──────────────────────────────────────────────────────────────────────
# Patch 2: Conv1D class (GPT-2 style)
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "Conv1D"):
    class Conv1D(nn.Module):
        def __init__(self, nf, nx):
            super().__init__()
            self.nf = nf
            self.weight = nn.Parameter(torch.empty(nx, nf))
            self.bias = nn.Parameter(torch.zeros(nf))
            nn.init.normal_(self.weight, std=0.02)
        
        def forward(self, x):
            size_out = x.size()[:-1] + (self.nf,)
            x = torch.addmm(self.bias, x.view(-1, x.size(-1)), self.weight)
            x = x.view(*size_out)
            return x
    _pu.Conv1D = Conv1D
    print("   ✅ Patched: Conv1D")

# ──────────────────────────────────────────────────────────────────────
# Patch 3: prune_conv1d_layer
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "prune_conv1d_layer"):
    def prune_conv1d_layer(layer, index, dim=1):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        if dim == 0:
            b = layer.bias.clone().detach()
        else:
            b = layer.bias[index].clone().detach()
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = _pu.Conv1D(new_size[1], new_size[0]).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        new_layer.bias.requires_grad = False
        new_layer.bias.copy_(b.contiguous())
        new_layer.bias.requires_grad = True
        return new_layer
    _pu.prune_conv1d_layer = prune_conv1d_layer
    print("   ✅ Patched: prune_conv1d_layer")

# ──────────────────────────────────────────────────────────────────────
# Patch 4: prune_linear_layer
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "prune_linear_layer"):
    def prune_linear_layer(layer, index, dim=0):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        if layer.bias is not None:
            if dim == 1:
                b = layer.bias.clone().detach()
            else:
                b = layer.bias[index].clone().detach()
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        if layer.bias is not None:
            new_layer.bias.requires_grad = False
            new_layer.bias.copy_(b.contiguous())
            new_layer.bias.requires_grad = True
        return new_layer
    _pu.prune_linear_layer = prune_linear_layer
    print("   ✅ Patched: prune_linear_layer")

# ──────────────────────────────────────────────────────────────────────
# Patch 5: prune_layer
# ──────────────────────────────────────────────────────────────────────
if not hasattr(_pu, "prune_layer"):
    def prune_layer(layer, index, dim=None):
        if isinstance(layer, _pu.Conv1D):
            if dim is None:
                dim = 1
            return _pu.prune_conv1d_layer(layer, index, dim=dim)
        elif isinstance(layer, nn.Linear):
            if dim is None:
                dim = 0
            return _pu.prune_linear_layer(layer, index, dim=dim)
        else:
            raise ValueError(f"Can't prune layer of class {layer.__class__}")
    _pu.prune_layer = prune_layer
    print("   ✅ Patched: prune_layer")

# ──────────────────────────────────────────────────────────────────────
# Patch 6: model_parallel_utils module (REMOVED in transformers 5.x)
# ──────────────────────────────────────────────────────────────────────
if 'transformers.utils.model_parallel_utils' not in sys.modules:
    mp_utils = types.ModuleType('transformers.utils.model_parallel_utils')
    
    def assert_device_map(device_map, num_blocks):
        blocks = list(range(num_blocks))
        device_map_blocks = [item for sublist in device_map.values() for item in sublist]
        
        duplicate_blocks = []
        for i in device_map_blocks:
            if device_map_blocks.count(i) > 1 and i not in duplicate_blocks:
                duplicate_blocks.append(i)
        
        missing_blocks = [i for i in blocks if i not in device_map_blocks]
        extra_blocks = [i for i in device_map_blocks if i not in blocks]
        
        if len(duplicate_blocks) != 0:
            raise ValueError(f"Duplicate blocks: {duplicate_blocks}")
        if len(missing_blocks) != 0:
            raise ValueError(f"Missing blocks: {missing_blocks}")
        if len(extra_blocks) != 0:
            raise ValueError(f"Extra blocks: {extra_blocks}")
    
    def get_device_map(n_layers, devices):
        layers = list(range(n_layers))
        n_blocks = int(math.ceil(n_layers / len(devices)))
        layers_list = [layers[i : i + n_blocks] for i in range(0, n_layers, n_blocks)]
        return dict(zip(devices, layers_list))
    
    mp_utils.assert_device_map = assert_device_map
    mp_utils.get_device_map = get_device_map
    
    sys.modules['transformers.utils.model_parallel_utils'] = mp_utils
    transformers.utils.model_parallel_utils = mp_utils
    print("   ✅ Patched: model_parallel_utils module")

print("\n✅ All patches applied!")
print("   Now safe to load Jais models with trust_remote_code=True")

🔧 Patching transformers for Jais compatibility...
   Transformers version: 5.12.1


✅ All patches applied!
   Now safe to load Jais models with trust_remote_code=True


## 🔧 CELL 8: Distillation Trainer Class

Custom trainer that combines:
- Cross-entropy loss with true labels
- KL divergence loss with teacher's answers

In [47]:
def prepare_distillation_dataset(distill_data, tokenizer, max_length=1024):
    """
    Convert distillation data to training format
    Format: Instruction-following with teacher's answer as supervised signal
    """
    texts = []
    
    for item in distill_data:
        if item['type'] == 'MCQ':
            # Use teacher's answer letter as the target
            options = item['options']
            teacher_letter = item['teacher_answer']
            teacher_text = options.get(teacher_letter, options.get('A', ''))
            
            instruction = f"""أنت خبير في القانون السعودي. أجب عن السؤال التالي.

السؤال: {item['question']}

الخيارات:
A) {options['A']}
B) {options['B']}
C) {options['C']}
D) {options['D']}

اختر الإجابة الصحيحة:"""
            
            response = f" {teacher_letter}"
            full_text = instruction + response
            
        else:  # QA
            instruction = f"""أنت خبير في القانون السعودي. أجب عن السؤال التالي بإجابة مباشرة ومختصرة.

السؤال: {item['question']}

الإجابة:"""
            
            response = f" {item['teacher_answer']}"
            full_text = instruction + response
        
        texts.append(full_text)
    
    return texts

def distill_student(student_size, student_model_name, distillation_data):
    """
    Distill knowledge from FANAR teacher into student model
    
    Uses task-specific distillation:
    - Trains student on teacher's answers
    - Standard cross-entropy loss
    - LoRA for efficiency
    """
    print("="*70)
    print(f"🎓 DISTILLING {student_size} STUDENT: {student_model_name}")
    print("="*70)
    
    # Load student tokenizer
    student_tokenizer = AutoTokenizer.from_pretrained(
        student_model_name,
        trust_remote_code=True,
        padding_side="left"
    )
    if student_tokenizer.pad_token is None:
        student_tokenizer.pad_token = student_tokenizer.eos_token
    
    # Load student in 4-bit for memory efficiency
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    
    print(f"📥 Loading student base model ({student_size})...")
    student_model = AutoModelForCausalLM.from_pretrained(
        student_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    
    student_info = get_model_size_info(student_model)
    print(f"   Parameters: {student_info['total_params_billions']:.2f}B")
    print(f"   Memory: {student_info['memory_gb']:.2f} GB")
    
    # Prepare for k-bit training
    student_model = prepare_model_for_kbit_training(student_model)
    
    # Apply LoRA
    lora_config = LoraConfig(
        r=Config.LORA_R,
        lora_alpha=Config.LORA_ALPHA,
        lora_dropout=Config.LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=Config.LORA_TARGET_MODULES,
    )
    
    student_model = get_peft_model(student_model, lora_config)
    print(f"\n📊 LoRA setup:")
    student_model.print_trainable_parameters()
    
    # Prepare distillation dataset
    print("\n📝 Preparing distillation dataset...")
    texts = prepare_distillation_dataset(distillation_data, student_tokenizer)
    
    def tokenize_fn(examples):
        return student_tokenizer(
            examples['text'],
            truncation=True,
            max_length=Config.MAX_LENGTH,
            padding=False,
        )
    
    train_dataset = HFDataset.from_dict({'text': texts})
    train_dataset = train_dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=['text']
    )
    
    print(f"   Training samples: {len(train_dataset)}")
    
    # Data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=student_tokenizer,
        mlm=False,
    )
    
    # Training arguments
    output_dir = f"{Config.MODELS_DIR}/distilled_{student_size}"
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=Config.NUM_EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION,
        learning_rate=Config.LEARNING_RATE,
        warmup_ratio=Config.WARMUP_RATIO,
        logging_steps=20,
        save_strategy="epoch",
        save_total_limit=1,
        bf16=True,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        report_to="none",
        seed=Config.SEED,
        remove_unused_columns=False,
    )
    
    # Standard trainer (distillation through teacher's labels)
    trainer = Trainer(
        model=student_model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
        tokenizer=student_tokenizer,
    )
    
    print(f"\n🚀 Starting distillation training...")
    print(f"   Epochs: {Config.NUM_EPOCHS}")
    print(f"   Effective batch size: {Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION}")
    
    trainer.train()
    
    # Save
    trainer.save_model(output_dir)
    student_tokenizer.save_pretrained(output_dir)
    
    print(f"\n✅ Distillation complete! Saved to: {output_dir}")
    
    return student_model, student_tokenizer, output_dir, student_info

print("✅ Distillation function defined")

✅ Distillation function defined


In [48]:
def distill_student(student_size, student_model_name, distillation_data):
    """
    Distill from Jais teacher into Qwen student
    Task-specific distillation (works cross-architecture)
    """
    print("="*70)
    print(f"🎓 DISTILLING {student_size} STUDENT: {student_model_name}")
    print("="*70)
    
    student_tokenizer = AutoTokenizer.from_pretrained(
        student_model_name,
        trust_remote_code=True,
        padding_side="left"
    )
    if student_tokenizer.pad_token is None:
        student_tokenizer.pad_token = student_tokenizer.eos_token
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    print(f"📥 Loading student base model ({student_size})...")
    student_model = AutoModelForCausalLM.from_pretrained(
        student_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
    
    student_info = get_model_size_info(student_model)
    print(f"   Parameters: {student_info['total_params_billions']:.2f}B")
    print(f"   Memory: {student_info['memory_gb']:.2f} GB")
    
    student_model = prepare_model_for_kbit_training(student_model)
    
    lora_config = LoraConfig(
        r=Config.LORA_R,
        lora_alpha=Config.LORA_ALPHA,
        lora_dropout=Config.LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=Config.LORA_TARGET_MODULES,
    )
    
    student_model = get_peft_model(student_model, lora_config)
    print(f"\n📊 LoRA setup:")
    student_model.print_trainable_parameters()
    
    print("\n📝 Preparing distillation dataset...")
    texts = prepare_distillation_dataset(distillation_data, student_tokenizer)
    
    def tokenize_fn(examples):
        return student_tokenizer(
            examples['text'],
            truncation=True,
            max_length=Config.MAX_LENGTH,
            padding=False,
        )
    
    train_dataset = HFDataset.from_dict({'text': texts})
    train_dataset = train_dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=['text']
    )
    
    print(f"   Training samples: {len(train_dataset)}")
    
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=student_tokenizer,
        mlm=False,
    )
    
    output_dir = f"{Config.MODELS_DIR}/distilled_{student_size}"
    
    total_steps = (len(train_dataset) // (Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION)) * Config.NUM_EPOCHS
    warmup_steps = int(total_steps * Config.WARMUP_RATIO)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=Config.NUM_EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION,
        learning_rate=Config.LEARNING_RATE,
        warmup_steps=warmup_steps,
        logging_steps=20,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        bf16=False,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        report_to="none",
        seed=Config.SEED,
        remove_unused_columns=False,
        gradient_checkpointing=True,
    )
    
    try:
        trainer = Trainer(
            model=student_model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            processing_class=student_tokenizer,
        )
    except TypeError:
        trainer = Trainer(
            model=student_model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            tokenizer=student_tokenizer,
        )
    
    print(f"\n🚀 Starting distillation training...")
    print(f"   Total steps: {total_steps}")
    print(f"   Warmup steps: {warmup_steps}")
    print(f"   Effective batch size: {Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION}")
    
    trainer.train()
    
    trainer.save_model(output_dir)
    student_tokenizer.save_pretrained(output_dir)
    
    print(f"\n✅ Distillation complete! Saved to: {output_dir}")
    
    return student_model, student_tokenizer, output_dir, student_info

print("✅ Distillation function defined (clean, no Jais hacks)")

✅ Distillation function defined (clean, no Jais hacks)


## 🎓 CELL 9: DISTILL TO 500M (Smallest First - Faster)

Start with smallest model for quick validation

In [49]:
#!pip install --upgrade transformers peft

In [50]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

student_size = '500M'
student_model_name = Config.STUDENT_MODELS[student_size]

model_500M, tokenizer_500M, path_500M, info_500M = distill_student(
    student_size, student_model_name, distillation_data
)

print(f"\n✅ 500M student distilled successfully")

🎓 DISTILLING 500M STUDENT: Qwen/Qwen2.5-0.5B-Instruct
📥 Loading student base model (500M)...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 377.08it/s]


   Parameters: 0.32B
   Memory: 0.42 GB


ValueError: Target modules {'c_fc', 'c_proj', 'c_attn'} not found in the base model. Please check the target modules and try again.

## 📊 CELL 10: Evaluate 500M Distilled Model

In [27]:
results_500M = evaluate_dual_setting(
    model_500M, tokenizer_500M, mcq_eval_sample,
    model_name="Distilled_500M"
)
results_500M['size_info'] = info_500M           # ← FIXED: 500M not 590M
results_500M['student_size'] = '500M'
results_500M['base_model'] = Config.STUDENT_MODELS['500M']

with open(f"{Config.RESULTS_DIR}/distilled_500M_results.json", 'w') as f:
    json.dump(results_500M, f, indent=2, default=str)  # ← FIXED: 500M

print(f"\n📊 500M DISTILLED RESULTS:")
print(f"   With Context:    {results_500M['with_context']['accuracy']:.2f}%")
print(f"   Without Context: {results_500M['without_context']['accuracy']:.2f}%")
print(f"   Memory:          {info_500M['memory_gb']:.2f} GB")

# Cleanup
del model_500M, tokenizer_500M                  # ← FIXED: 500M not 590M
gc.collect()
torch.cuda.empty_cache()

NameError: name 'model_500M' is not defined

## 🎓 CELL 11: DISTILL TO 1.5B

In [28]:
student_size = '1.5B'
student_model_name = Config.STUDENT_MODELS[student_size]

model_1_5B, tokenizer_1_5B, path_1_5B, info_1_5B = distill_student(
    student_size, student_model_name, distillation_data
)

print(f"\n✅ 1.5B student distilled successfully")

KeyError: '1.5B'

## 📊 CELL 12: Evaluate 1.5B Distilled Model

In [108]:
results_1_5B = evaluate_dual_setting(
    model_1_5B, tokenizer_1_5B, mcq_eval_sample,
    model_name="Distilled_1.5B"
)
results_1_5B['size_info'] = info_1_5B
results_1_5B['student_size'] = '1.5B'
results_1_5B['base_model'] = Config.STUDENT_MODELS['1.5B']

with open(f"{Config.RESULTS_DIR}/distilled_1.5B_results.json", 'w') as f:
    json.dump(results_1_5B, f, indent=2, default=str)

print(f"\n📊 1.5B DISTILLED RESULTS:")
print(f"   With Context:    {results_1_5B['with_context']['accuracy']:.2f}%")
print(f"   Without Context: {results_1_5B['without_context']['accuracy']:.2f}%")
print(f"   Memory:          {info_1_5B['memory_gb']:.2f} GB")

# Free memory
del model_1_5B, tokenizer_1_5B
cleanup_memory()


📊 Evaluating: Distilled_1.5B


Distilled_1.5B (no ctx): 100%|██████████| 500/500 [07:44<00:00,  1.08it/s]



📊 1.5B DISTILLED RESULTS:
   With Context:    90.80%
   Without Context: 66.40%
   Memory:          1.05 GB
   GPU Memory: 0.02 GB


## 🎓 CELL 13: DISTILL TO 3B

In [109]:
student_size = '3B'
student_model_name = Config.STUDENT_MODELS[student_size]

model_3B, tokenizer_3B, path_3B, info_3B = distill_student(
    student_size, student_model_name, distillation_data
)

print(f"\n✅ 3B student distilled successfully")

🎓 DISTILLING 3B STUDENT: Qwen/Qwen2.5-3B-Instruct
📥 Loading student base model (3B)...


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 167.71it/s]


   Parameters: 1.70B
   Memory: 1.87 GB

📊 LoRA setup:
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607

📝 Preparing distillation dataset...


Map: 100%|██████████| 10251/10251 [00:00<00:00, 21351.41 examples/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


   Training samples: 10251

🚀 Starting distillation training...
   Total steps: 1920
   Warmup steps: 192
   Effective batch size: 16


Step,Training Loss
20,2.014086
40,1.499292
60,1.245956
80,1.191207
100,1.143635
120,1.096777
140,1.090260
160,1.074037
180,1.070793
200,1.080361



✅ Distillation complete! Saved to: ./models_jais_stage3/distilled_3B

✅ 3B student distilled successfully


## 📊 CELL 14: Evaluate 3B Distilled Model

In [110]:
results_3B = evaluate_dual_setting(
    model_3B, tokenizer_3B, mcq_eval_sample,
    model_name="Distilled_3B"
)
results_3B['size_info'] = info_3B
results_3B['student_size'] = '3B'
results_3B['base_model'] = Config.STUDENT_MODELS['3B']

with open(f"{Config.RESULTS_DIR}/distilled_3B_results.json", 'w') as f:
    json.dump(results_3B, f, indent=2, default=str)

print(f"\n📊 3B DISTILLED RESULTS:")
print(f"   With Context:    {results_3B['with_context']['accuracy']:.2f}%")
print(f"   Without Context: {results_3B['without_context']['accuracy']:.2f}%")
print(f"   Memory:          {info_3B['memory_gb']:.2f} GB")

# Free memory
del model_3B, tokenizer_3B
cleanup_memory()


📊 Evaluating: Distilled_3B


Distilled_3B (no ctx): 100%|██████████| 500/500 [09:52<00:00,  1.18s/it]



📊 3B DISTILLED RESULTS:
   With Context:    94.40%
   Without Context: 70.20%
   Memory:          1.87 GB
   GPU Memory: 0.02 GB


## 🎓 CELL 15: DISTILL TO 7B (Largest Student)

In [21]:
# ──────────────────────────────────────────────────────────────────────
# Patch 6: model_parallel_utils module (removed in transformers 5.x)
# Jais needs assert_device_map and get_device_map from this module
# ──────────────────────────────────────────────────────────────────────
import sys
import types

if 'transformers.utils.model_parallel_utils' not in sys.modules:
    # Create the missing module
    mp_utils = types.ModuleType('transformers.utils.model_parallel_utils')
    
    def assert_device_map(device_map, num_blocks):
        """Verify device_map is valid for the model."""
        blocks = list(range(num_blocks))
        device_map_blocks = [item for sublist in device_map.values() for item in sublist]
        
        # Duplicates check
        duplicate_blocks = []
        for i in device_map_blocks:
            if device_map_blocks.count(i) > 1 and i not in duplicate_blocks:
                duplicate_blocks.append(i)
        
        missing_blocks = [i for i in blocks if i not in device_map_blocks]
        extra_blocks = [i for i in device_map_blocks if i not in blocks]
        
        if len(duplicate_blocks) != 0:
            raise ValueError(
                "Duplicate attention blocks specified in device_map. Attention blocks "
                f"{duplicate_blocks} are repeated."
            )
        if len(missing_blocks) != 0:
            raise ValueError(
                f"There are attention blocks for this model that are not specified in the device_map. "
                f"Add these attention blocks {missing_blocks} to a device."
            )
        if len(extra_blocks) != 0:
            raise ValueError(
                f"The device_map contains more attention blocks than this model has. "
                f"Remove these from the device_map: {extra_blocks}"
            )
    
    def get_device_map(n_layers, devices):
        """Returns a dictionary of layers distributed evenly across all devices."""
        layers = list(range(n_layers))
        n_blocks = int(math.ceil(n_layers / len(devices)))
        layers_list = [layers[i : i + n_blocks] for i in range(0, n_layers, n_blocks)]
        return dict(zip(devices, layers_list))
    
    import math
    mp_utils.assert_device_map = assert_device_map
    mp_utils.get_device_map = get_device_map
    
    sys.modules['transformers.utils.model_parallel_utils'] = mp_utils
    
    # Also make it accessible via transformers.utils
    import transformers.utils
    transformers.utils.model_parallel_utils = mp_utils
    
    print("   ✅ Patched: transformers.utils.model_parallel_utils module")
else:
    print("   ℹ️  model_parallel_utils already present")

print("\n✅ All patches complete!")

   ✅ Patched: transformers.utils.model_parallel_utils module

✅ All patches complete!


In [23]:
# ──────────────────────────────────────────────────────────────────────
# Patch 7: Add missing config attributes for JAISConfig
# ──────────────────────────────────────────────────────────────────────
from transformers import PretrainedConfig

_original_init = PretrainedConfig.__init__

def _patched_init(self, *args, **kwargs):
    _original_init(self, *args, **kwargs)
    # Add common attributes that newer Jais models expect
    if not hasattr(self, 'add_cross_attention'):
        self.add_cross_attention = False
    if not hasattr(self, 'cross_attention_hidden_size'):
        self.cross_attention_hidden_size = None
    if not hasattr(self, 'is_encoder_decoder'):
        self.is_encoder_decoder = False
    if not hasattr(self, 'is_decoder'):
        self.is_decoder = False
    if not hasattr(self, 'chunk_size_feed_forward'):
        self.chunk_size_feed_forward = 0

PretrainedConfig.__init__ = _patched_init
print("   ✅ Patched: PretrainedConfig attributes (add_cross_attention, etc.)")

   ✅ Patched: PretrainedConfig attributes (add_cross_attention, etc.)


In [26]:
!pip install -U bitsandbytes

Defaulting to user installation because normal site-packages is not writeable


In [28]:
def distill_student_fp16(student_size, student_model_name, distillation_data):
    """
    Distill student WITHOUT 4-bit quantization (for compatibility)
    Uses FP16 directly - works around bitsandbytes errors
    """
    print("="*70)
    print(f"🎓 DISTILLING {student_size} STUDENT: {student_model_name}")
    print(f"   Mode: FP16 (no 4-bit quantization)")
    print("="*70)
    
    student_tokenizer = AutoTokenizer.from_pretrained(
        student_model_name,
        trust_remote_code=True,
        padding_side="left"
    )
    if student_tokenizer.pad_token is None:
        student_tokenizer.pad_token = student_tokenizer.eos_token
    
    # Load DIRECTLY in FP16 (no quantization config)
    print(f"📥 Loading student base model ({student_size}) in FP16...")
    student_model = AutoModelForCausalLM.from_pretrained(
        student_model_name,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    
    student_info = get_model_size_info(student_model)
    print(f"   Parameters: {student_info['total_params_billions']:.2f}B")
    print(f"   Memory: {student_info['memory_gb']:.2f} GB")
    
    # Apply LoRA directly (no kbit prep needed for FP16)
    lora_config = LoraConfig(
        r=Config.LORA_R,
        lora_alpha=Config.LORA_ALPHA,
        lora_dropout=Config.LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=Config.LORA_TARGET_MODULES,
    )
    
    student_model = get_peft_model(student_model, lora_config)
    print(f"\n📊 LoRA setup:")
    student_model.print_trainable_parameters()
    
    # Prepare dataset
    print("\n📝 Preparing distillation dataset...")
    texts = prepare_distillation_dataset(distillation_data, student_tokenizer)
    
    def tokenize_fn(examples):
        return student_tokenizer(
            examples['text'],
            truncation=True,
            max_length=Config.MAX_LENGTH,
            padding=False,
        )
    
    train_dataset = HFDataset.from_dict({'text': texts})
    train_dataset = train_dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=['text']
    )
    
    print(f"   Training samples: {len(train_dataset)}")
    
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=student_tokenizer,
        mlm=False,
    )
    
    output_dir = f"{Config.MODELS_DIR}/distilled_{student_size}"
    
    total_steps = (len(train_dataset) // (Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION)) * Config.NUM_EPOCHS
    warmup_steps = int(total_steps * Config.WARMUP_RATIO)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=Config.NUM_EPOCHS,
        per_device_train_batch_size=1,  # Smaller batch for FP16 7B
        gradient_accumulation_steps=16,  # Keep effective batch = 16
        learning_rate=Config.LEARNING_RATE,
        warmup_steps=warmup_steps,
        logging_steps=20,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        bf16=False,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        report_to="none",
        seed=Config.SEED,
        remove_unused_columns=False,
        gradient_checkpointing=True,
    )
    
    try:
        trainer = Trainer(
            model=student_model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            processing_class=student_tokenizer,
        )
    except TypeError:
        trainer = Trainer(
            model=student_model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            tokenizer=student_tokenizer,
        )
    
    print(f"\n🚀 Starting distillation training...")
    print(f"   Total steps: {total_steps}")
    
    trainer.train()
    
    trainer.save_model(output_dir)
    student_tokenizer.save_pretrained(output_dir)
    
    print(f"\n✅ Distillation complete! Saved to: {output_dir}")
    
    return student_model, student_tokenizer, output_dir, student_info

print("✅ FP16 distillation function defined (no quantization)")

✅ FP16 distillation function defined (no quantization)


In [30]:
import torch

if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free = torch.cuda.mem_get_info()[0] / 1e9
    
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total memory: {total:.2f} GB")
    print(f"Free memory: {free:.2f} GB")
    
    if free < 22:
        print(f"\n⚠️ WARNING: 7B FP16 needs ~20GB")
        print(f"   Consider: skip 7B OR fix bitsandbytes")
    else:
        print(f"\n✅ Should fit for 7B FP16 training")

GPU: NVIDIA A100-PCIE-40GB
Total memory: 42.41 GB
Free memory: 21.71 GB

⚠️ WARNING: 7B FP16 needs ~20GB
   Consider: skip 7B OR fix bitsandbytes


In [31]:
import gc
import torch

# Aggressive cleanup
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Try to free more
try:
    # Delete any model variables that might exist
    for var in ['model_500M', 'model_1_5B', 'model_3B', 'model_7B',
                'tokenizer_500M', 'tokenizer_1_5B', 'tokenizer_3B', 'tokenizer_7B',
                'student_model', 'teacher_model', 'trainer']:
        if var in dir():
            exec(f"del {var}")
except:
    pass

gc.collect()
torch.cuda.empty_cache()

# Check memory after cleanup
import torch
total = torch.cuda.get_device_properties(0).total_memory / 1e9
free = torch.cuda.mem_get_info()[0] / 1e9
allocated = torch.cuda.memory_allocated() / 1e9

print(f"After cleanup:")
print(f"   Total: {total:.2f} GB")
print(f"   Free:  {free:.2f} GB")
print(f"   Used:  {allocated:.2f} GB")

After cleanup:
   Total: 42.41 GB
   Free:  26.47 GB
   Used:  10.70 GB


In [32]:
import os

print("Stage 3 Distillation Status:")
for size in ['500M', '1.5B', '3B']:
    model_path = f"./models_jais_stage3/distilled_{size}"
    results_path = f"./results_jais_stage3/distilled_{size}_results.json"
    
    has_model = os.path.exists(model_path)
    has_results = os.path.exists(results_path)
    
    status = "✅" if (has_model and has_results) else ("🔄" if has_model else "❌")
    print(f"   {status} {size}: model={has_model}, results={has_results}")

Stage 3 Distillation Status:
   ✅ 500M: model=True, results=True
   ✅ 1.5B: model=True, results=True
   ✅ 3B: model=True, results=True


In [33]:
student_size = '7B'
student_model_name = Config.STUDENT_MODELS[student_size]

model_7B, tokenizer_7B, path_7B, info_7B = distill_student(
    student_size, student_model_name, distillation_data
)

print(f"\n✅ 7B student distilled successfully")

🎓 DISTILLING 7B STUDENT: inceptionai/jais-family-6p7b-chat


[transformers] JAISLMHeadModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


📥 Loading student base model (7B)...


Loading weights: 100%|██████████| 453/453 [00:06<00:00, 72.24it/s]


RuntimeError: "normal_kernel_cuda" not implemented for 'Byte'

## 📊 CELL 16: Evaluate 7B Distilled Model

In [ ]:
results_7B = evaluate_dual_setting(
    model_7B, tokenizer_7B, mcq_eval_sample,
    model_name="Distilled_7B"
)
results_7B['size_info'] = info_7B
results_7B['student_size'] = '7B'
results_7B['base_model'] = Config.STUDENT_MODELS['7B']

with open(f"{Config.RESULTS_DIR}/distilled_7B_results.json", 'w') as f:
    json.dump(results_7B, f, indent=2, default=str)

print(f"\n📊 7B DISTILLED RESULTS:")
print(f"   With Context:    {results_7B['with_context']['accuracy']:.2f}%")
print(f"   Without Context: {results_7B['without_context']['accuracy']:.2f}%")
print(f"   Memory:          {info_7B['memory_gb']:.2f} GB")

# Free memory
del model_7B, tokenizer_7B
cleanup_memory()

## 📊 CELL 17: Combine All Distillation Results

In [ ]:
# Compile all distilled model results
all_distilled = {
    '590M': results_590M,
    '1.5B': results_1_5B,
    '3B': results_3B,
    '7B': results_7B,
}

# Create comparison dataframe
distill_comparison = []
for size, results in all_distilled.items():
    distill_comparison.append({
        'Student Size': size,
        'Base Model': results['base_model'].split('/')[-1],
        'Parameters (B)': round(results['size_info']['total_params_billions'], 2),
        'Memory (GB)': round(results['size_info']['memory_gb'], 2),
        'With Context (%)': round(results['with_context']['accuracy'], 2),
        'Without Context (%)': round(results['without_context']['accuracy'], 2),
        'Reading Skill': round(results['reading_skill'], 2),
    })

distill_df = pd.DataFrame(distill_comparison)

# Add teacher reference row
teacher_row = pd.DataFrame([{
    'Student Size': '9B (Teacher)',
    'Base Model': 'FANAR-9B-SFT',
    'Parameters (B)': 9.0,
    'Memory (GB)': 16.36,
    'With Context (%)': 99.00,
    'Without Context (%)': 79.20,
    'Reading Skill': 19.80,
}])

distill_df = pd.concat([teacher_row, distill_df], ignore_index=True)

print("="*100)
print("📊 STAGE 3: DISTILLATION RESULTS")
print("="*100)
print(distill_df.to_string(index=False))

distill_df.to_csv(f"{Config.RESULTS_DIR}/stage3_distillation_comparison.csv", index=False)
print(f"\n✅ Saved to: {Config.RESULTS_DIR}/stage3_distillation_comparison.csv")

## 🔬 CELL 18: PHASE 2 - Apply INT4 Quantization to Each Distilled Model

Use Stage 2's best method (INT4 NF4) on all distilled students

In [ ]:
def quantize_and_evaluate_distilled(student_size, lora_path):
    """
    Load distilled model, merge LoRA, quantize to INT4 NF4, evaluate
    """
    print("="*70)
    print(f"🔬 QUANTIZING DISTILLED {student_size} TO INT4 NF4")
    print("="*70)
    
    base_model_name = Config.STUDENT_MODELS[student_size]
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name,
        trust_remote_code=True,
        padding_side="left"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load base model in BF16 first
    print(f"📥 Loading base + LoRA...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    
    # Apply LoRA and merge
    model = PeftModel.from_pretrained(base_model, lora_path)
    model = model.merge_and_unload()
    
    # Save merged model
    merged_path = f"{Config.MODELS_DIR}/distilled_{student_size}_merged"
    model.save_pretrained(merged_path)
    tokenizer.save_pretrained(merged_path)
    
    # Free memory
    del model, base_model
    cleanup_memory()
    
    # Load in INT4 NF4
    print(f"📥 Loading in INT4 NF4...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    
    quantized_model = AutoModelForCausalLM.from_pretrained(
        merged_path,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    quantized_model.eval()
    
    info = get_model_size_info(quantized_model)
    print(f"   Quantized memory: {info['memory_gb']:.2f} GB")
    
    # Evaluate
    results = evaluate_dual_setting(
        quantized_model, tokenizer, mcq_eval_sample,
        model_name=f"Distilled_{student_size}_INT4"
    )
    results['size_info'] = info
    results['student_size'] = student_size
    results['quantization'] = 'INT4_NF4'
    
    # Save
    with open(f"{Config.RESULTS_DIR}/distilled_{student_size}_INT4_results.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print(f"\n📊 {student_size} INT4 Results:")
    print(f"   With Context:    {results['with_context']['accuracy']:.2f}%")
    print(f"   Without Context: {results['without_context']['accuracy']:.2f}%")
    print(f"   Memory:          {info['memory_gb']:.2f} GB")
    
    # Cleanup
    del quantized_model, tokenizer
    cleanup_memory()
    
    return results

print("✅ Quantization function defined")

In [ ]:
# Quantize 500M
results_500M_INT4 = quantize_and_evaluate_distilled('500M', path_500M)

In [ ]:
# Quantize 1.5B
results_1_5B_INT4 = quantize_and_evaluate_distilled('1.5B', path_1_5B)

In [ ]:
# Quantize 3B
results_3B_INT4 = quantize_and_evaluate_distilled('3B', path_3B)

In [ ]:
# Quantize 7B
results_7B_INT4 = quantize_and_evaluate_distilled('7B', path_7B)

## 📊 CELL 19: Build Complete Pareto Frontier

In [ ]:
# Compile ALL results: Stage 1 + Stage 2 + Stage 3
pareto_data = [
    # Stage 1 Teacher (BF16)
    {
        'Model': 'FANAR-9B-SFT (Teacher)',
        'Size': '9B',
        'Source': 'Stage 1',
        'Method': 'Original',
        'Precision': 'BF16',
        'Memory_GB': 16.36,
        'WithContext': 99.00,
        'WithoutContext': 79.20,
    },
    # Stage 2 Quantizations of teacher
    {
        'Model': 'FANAR-9B INT4',
        'Size': '9B',
        'Source': 'Stage 2',
        'Method': 'Quantization',
        'Precision': 'INT4_NF4',
        'Memory_GB': 4.73,
        'WithContext': 99.00,  # Replace with your actual Stage 2 numbers
        'WithoutContext': 79.40,
    },
    {
        'Model': 'FANAR-9B INT8',
        'Size': '9B',
        'Source': 'Stage 2',
        'Method': 'Quantization',
        'Precision': 'INT8',
        'Memory_GB': 8.61,
        'WithContext': 99.00,
        'WithoutContext': 79.40,
    },
]

# Add Stage 3 distilled models (BF16)
for size, results in [('7B', results_7B), ('3B', results_3B), ('1.5B', results_1_5B), ('500M', results_500M)]:
    pareto_data.append({
        'Model': f'Distilled-{size}',
        'Size': size,
        'Source': 'Stage 3',
        'Method': 'Distillation',
        'Precision': 'BF16',
        'Memory_GB': round(results['size_info']['memory_gb'], 2),
        'WithContext': round(results['with_context']['accuracy'], 2),
        'WithoutContext': round(results['without_context']['accuracy'], 2),
    })

# Add Stage 3 distilled + quantized models
for size, results in [('7B', results_7B_INT4), ('3B', results_3B_INT4), 
                       ('1.5B', results_1_5B_INT4), ('500M', results_500M_INT4)]:
    pareto_data.append({
        'Model': f'Distilled-{size}-INT4',
        'Size': size,
        'Source': 'Stage 3',
        'Method': 'Distill+Quantize',
        'Precision': 'INT4_NF4',
        'Memory_GB': round(results['size_info']['memory_gb'], 2),
        'WithContext': round(results['with_context']['accuracy'], 2),
        'WithoutContext': round(results['without_context']['accuracy'], 2),
    })

pareto_df = pd.DataFrame(pareto_data)
pareto_df = pareto_df.sort_values('Memory_GB', ascending=False).reset_index(drop=True)

print("="*100)
print("📊 COMPLETE PARETO FRONTIER: All Models from Stages 1, 2, 3")
print("="*100)
print(pareto_df.to_string(index=False))

pareto_df.to_csv(f"{Config.RESULTS_DIR}/pareto_frontier_complete.csv", index=False)
print(f"\n✅ Saved: {Config.RESULTS_DIR}/pareto_frontier_complete.csv")

## 📈 CELL 20: Visualize Pareto Frontier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Color by method
method_colors = {
    'Original': '#3D5A80',
    'Quantization': '#EE6C4D',
    'Distillation': '#98C1D9',
    'Distill+Quantize': '#E63946',
}

# Plot 1: Memory vs Without-Context Accuracy
for method in pareto_df['Method'].unique():
    subset = pareto_df[pareto_df['Method'] == method]
    axes[0].scatter(subset['Memory_GB'], subset['WithoutContext'],
                    s=200, c=method_colors[method], edgecolor='black',
                    label=method, alpha=0.8, zorder=3)
    
    for idx, row in subset.iterrows():
        axes[0].annotate(
            f"{row['Size']}",
            xy=(row['Memory_GB'], row['WithoutContext']),
            xytext=(7, 7), textcoords='offset points',
            fontsize=9, fontweight='bold'
        )

axes[0].set_xlabel('Memory Footprint (GB)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy Without Context (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Pareto Frontier: Memory vs Real Knowledge', fontsize=14, fontweight='bold')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='lower right', fontsize=10)

# Plot 2: Memory vs With-Context Accuracy
for method in pareto_df['Method'].unique():
    subset = pareto_df[pareto_df['Method'] == method]
    axes[1].scatter(subset['Memory_GB'], subset['WithContext'],
                    s=200, c=method_colors[method], edgecolor='black',
                    label=method, alpha=0.8, zorder=3)
    
    for idx, row in subset.iterrows():
        axes[1].annotate(
            f"{row['Size']}",
            xy=(row['Memory_GB'], row['WithContext']),
            xytext=(7, 7), textcoords='offset points',
            fontsize=9, fontweight='bold'
        )

axes[1].set_xlabel('Memory Footprint (GB)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy With Context (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Pareto Frontier: Memory vs Reading Comprehension', fontsize=14, fontweight='bold')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='lower right', fontsize=10)

plt.suptitle('Complete Pareto Frontier: Arabic Legal LLM Compression', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{Config.RESULTS_DIR}/pareto_frontier_main.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {Config.RESULTS_DIR}/pareto_frontier_main.png")

In [ ]:
# Plot 2: Performance Degradation by Size
fig, ax = plt.subplots(figsize=(12, 6))

sizes_order = ['9B', '7B', '3B', '1.5B', '500M']
size_to_params = {'9B': 9, '7B': 7, '3B': 3, '1.5B': 1.5, '500M': 0.5}

# BF16 line
bf16_data = pareto_df[pareto_df['Precision'] == 'BF16'].copy()
bf16_data['ParamsB'] = bf16_data['Size'].map(size_to_params)
bf16_data = bf16_data.sort_values('ParamsB', ascending=False)

# INT4 line
int4_data = pareto_df[pareto_df['Precision'] == 'INT4_NF4'].copy()
int4_data['ParamsB'] = int4_data['Size'].map(size_to_params)
int4_data = int4_data.sort_values('ParamsB', ascending=False)

ax.plot(bf16_data['ParamsB'], bf16_data['WithoutContext'], 
        'o-', color='#3D5A80', markersize=12, linewidth=2.5,
        label='BF16', markeredgecolor='black')
ax.plot(int4_data['ParamsB'], int4_data['WithoutContext'],
        's-', color='#EE6C4D', markersize=12, linewidth=2.5,
        label='INT4 NF4', markeredgecolor='black')

# Annotations
for _, row in bf16_data.iterrows():
    ax.annotate(f"{row['WithoutContext']:.1f}%",
                xy=(row['ParamsB'], row['WithoutContext']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold', color='#3D5A80')

for _, row in int4_data.iterrows():
    ax.annotate(f"{row['WithoutContext']:.1f}%",
                xy=(row['ParamsB'], row['WithoutContext']),
                xytext=(0, -18), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold', color='#EE6C4D')

# Mark performance cliff if exists
ax.axhline(y=50, color='red', linestyle='--', alpha=0.4, label='50% threshold')
ax.axvspan(0.5, 1.5, alpha=0.1, color='red', label='Potential cliff zone')

ax.set_xlabel('Model Size (Billion Parameters)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy Without Context (%)', fontsize=12, fontweight='bold')
ax.set_title('Performance vs Model Size: Finding the Performance Cliff', 
             fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(f'{Config.RESULTS_DIR}/performance_cliff_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved: {Config.RESULTS_DIR}/performance_cliff_analysis.png")

In [ ]:
# Plot 3: Compression Ratio Analysis
fig, ax = plt.subplots(figsize=(14, 7))

baseline_memory = 16.36
baseline_accuracy_no_ctx = 79.20

pareto_df['CompressionRatio'] = baseline_memory / pareto_df['Memory_GB']
pareto_df['AccuracyRetention'] = (pareto_df['WithoutContext'] / baseline_accuracy_no_ctx) * 100

for method in pareto_df['Method'].unique():
    subset = pareto_df[pareto_df['Method'] == method]
    ax.scatter(subset['CompressionRatio'], subset['AccuracyRetention'],
               s=300, c=method_colors[method], edgecolor='black',
               label=method, alpha=0.8, zorder=3)
    
    for idx, row in subset.iterrows():
        ax.annotate(
            f"{row['Size']}",
            xy=(row['CompressionRatio'], row['AccuracyRetention']),
            xytext=(8, 8), textcoords='offset points',
            fontsize=10, fontweight='bold'
        )

# Pareto frontier line
frontier = pareto_df.sort_values('CompressionRatio')
best_at_each_x = []
current_best = 0
for _, row in frontier.sort_values('CompressionRatio', ascending=False).iterrows():
    if row['AccuracyRetention'] > current_best:
        best_at_each_x.append(row)
        current_best = row['AccuracyRetention']

if best_at_each_x:
    frontier_df = pd.DataFrame(best_at_each_x).sort_values('CompressionRatio')
    ax.plot(frontier_df['CompressionRatio'], frontier_df['AccuracyRetention'],
            'k--', alpha=0.5, linewidth=2, label='Pareto Frontier')

ax.set_xlabel('Compression Ratio (X times smaller)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy Retention (%)', fontsize=12, fontweight='bold')
ax.set_title('Pareto Frontier: Compression vs Accuracy Retention', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower left', fontsize=11)
ax.axhline(y=100, color='green', linestyle=':', alpha=0.5, label='100% retention')

plt.tight_layout()
plt.savefig(f'{Config.RESULTS_DIR}/pareto_compression_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Saved: {Config.RESULTS_DIR}/pareto_compression_accuracy.png")

## 🏆 CELL 21: Final Stage 3 Report

In [ ]:
# Find optimal models
pareto_df['ParamsB'] = pareto_df['Size'].map(size_to_params)

# Best at different memory thresholds
best_under_1gb = pareto_df[pareto_df['Memory_GB'] < 1.0].nlargest(1, 'WithoutContext')
best_under_2gb = pareto_df[pareto_df['Memory_GB'] < 2.0].nlargest(1, 'WithoutContext')
best_under_5gb = pareto_df[pareto_df['Memory_GB'] < 5.0].nlargest(1, 'WithoutContext')

# Find performance cliff
sorted_by_size = pareto_df[pareto_df['Precision'] == 'BF16'].sort_values('ParamsB')
cliff_drops = []
for i in range(1, len(sorted_by_size)):
    prev = sorted_by_size.iloc[i-1]
    curr = sorted_by_size.iloc[i]
    drop = curr['WithoutContext'] - prev['WithoutContext']
    cliff_drops.append((prev['Size'], curr['Size'], drop))

report = f"""
{'='*70}
STAGE 3 FINAL REPORT - DISTILLATION + COMPLETE PARETO FRONTIER
{'='*70}

Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Teacher: FANAR-9B-SFT (Stage 1)
Students: Qwen2.5 family (7B, 3B, 1.5B, 0.5B)
Dataset: ArabLegalEval ({len(mcq_eval_sample)} eval samples)

{'='*70}
DISTILLATION RESULTS (BF16)
{'='*70}

{'Size':<10} {'Params':>10} {'Memory':>10} {'With Ctx':>10} {'No Ctx':>10}
{'-'*55}
{'9B (T)':<10} {'9.00B':>10} {'16.36 GB':>10} {'99.00%':>10} {'79.20%':>10}
"""

for size in ['7B', '3B', '1.5B', '500M']:
    r = all_distilled[size]
    params = r['size_info']['total_params_billions']
    memory = r['size_info']['memory_gb']
    with_c = r['with_context']['accuracy']
    without_c = r['without_context']['accuracy']
    report += f"{size:<10} {params:>9.2f}B {memory:>9.2f} GB {with_c:>9.2f}% {without_c:>9.2f}%\n"

report += f"""
{'='*70}
DISTILLATION + INT4 QUANTIZATION
{'='*70}

{'Size':<10} {'Memory':>10} {'With Ctx':>10} {'No Ctx':>10}
{'-'*45}
"""

all_int4 = {
    '7B': results_7B_INT4,
    '3B': results_3B_INT4,
    '1.5B': results_1_5B_INT4,
    '500M': results_500M_INT4,
}

for size in ['7B', '3B', '1.5B', '500M']:
    r = all_int4[size]
    memory = r['size_info']['memory_gb']
    with_c = r['with_context']['accuracy']
    without_c = r['without_context']['accuracy']
    report += f"{size:<10} {memory:>9.2f} GB {with_c:>9.2f}% {without_c:>9.2f}%\n"

report += f"""
{'='*70}
KEY FINDINGS
{'='*70}

🏆 BEST MODELS BY DEPLOYMENT CONSTRAINT:
"""

if not best_under_5gb.empty:
    row = best_under_5gb.iloc[0]
    report += f"""
   Under 5GB (Laptop):     {row['Model']}
                          Memory: {row['Memory_GB']:.2f} GB
                          Accuracy: {row['WithoutContext']:.2f}%
"""

if not best_under_2gb.empty:
    row = best_under_2gb.iloc[0]
    report += f"""
   Under 2GB (Mobile):     {row['Model']}
                          Memory: {row['Memory_GB']:.2f} GB
                          Accuracy: {row['WithoutContext']:.2f}%
"""

if not best_under_1gb.empty:
    row = best_under_1gb.iloc[0]
    report += f"""
   Under 1GB (Edge):       {row['Model']}
                          Memory: {row['Memory_GB']:.2f} GB
                          Accuracy: {row['WithoutContext']:.2f}%
"""

report += f"""
📉 PERFORMANCE DROPS BY SIZE:
"""

for prev_size, curr_size, drop in cliff_drops:
    indicator = "⚠️" if drop < -10 else "✓"
    report += f"   {indicator} {prev_size} → {curr_size}: {drop:+.2f}%\n"

report += f"""
{'='*70}
FILES SAVED
{'='*70}

Results: {Config.RESULTS_DIR}/
  - pareto_frontier_complete.csv
  - pareto_frontier_main.png
  - performance_cliff_analysis.png
  - pareto_compression_accuracy.png
  - distilled_*_results.json (8 files)
  - stage3_distillation_comparison.csv

Models: {Config.MODELS_DIR}/
  - distilled_500M/, distilled_1.5B/, distilled_3B/, distilled_7B/
  - distilled_*_merged/ (4 merged versions)

{'='*70}
NEXT STEPS - PAPER WRITING
{'='*70}

1. Use complete Pareto frontier in paper figures
2. Analyze performance cliffs in discussion
3. Compare distillation vs quantization trade-offs
4. Recommend optimal deployment models

{'='*70}
"""

print(report)

with open(f"{Config.RESULTS_DIR}/stage3_final_report.txt", 'w', encoding='utf-8') as f:
    f.write(report)

# Save consolidated results
consolidated = {
    'date': datetime.now().isoformat(),
    'teacher': Config.TEACHER_LORA_PATH,
    'pareto_frontier': pareto_df.to_dict(orient='records'),
    'distilled_models': {
        '500M': results_500M,
        '1.5B': results_1_5B,
        '3B': results_3B,
        '7B': results_7B,
    },
    'distilled_quantized': {
        '500M': results_500M_INT4,
        '1.5B': results_1_5B_INT4,
        '3B': results_3B_INT4,
        '7B': results_7B_INT4,
    },
}

with open(f"{Config.RESULTS_DIR}/stage3_all_results.json", 'w', encoding='utf-8') as f:
    json.dump(consolidated, f, ensure_ascii=False, indent=2, default=str)

print(f"\n✅ STAGE 3 COMPLETE!")
print(f"\n🎯 You now have:")
print(f"   - 4 distilled models (7B, 3B, 1.5B, 500M)")
print(f"   - 4 distilled + quantized models (INT4 NF4)")
print(f"   - Complete Pareto frontier (10+ data points)")
print(f"   - Ready for paper writing!")

## 🎉 STAGE 3 COMPLETE!

### Summary of Achievements:

✅ **Distilled 4 student models** at sizes: 7B, 3B, 1.5B, 500M

✅ **Applied INT4 NF4 quantization** to each distilled model

✅ **Built complete Pareto frontier** combining Stages 1, 2, and 3:
- 9B teacher (Stage 1)
- 9B quantized versions (Stage 2)
- 4 distilled sizes (Stage 3)
- 4 distilled + quantized (Stage 3 + Stage 2)

✅ **Generated publication-ready figures:**
- Pareto frontier (memory vs accuracy)
- Performance cliff analysis
- Compression vs accuracy retention

✅ **Identified deployment recommendations:**
- Best model under 5GB (laptop)
- Best model under 2GB (mobile)
- Best model under 1GB (edge)

### Output Files Ready for Paper:

**Tables (CSV):**
- `pareto_frontier_complete.csv`
- `stage3_distillation_comparison.csv`

**Figures (PNG):**
- `pareto_frontier_main.png` - Main Pareto plot
- `performance_cliff_analysis.png` - Size vs accuracy
- `pareto_compression_accuracy.png` - Compression analysis

**Reports:**
- `stage3_final_report.txt` - Text summary
- `stage3_all_results.json` - All raw data

### Next Steps:

1. **Write paper** using these results
2. **Target venues:** EMNLP Findings, ACL Findings, ArabicNLP
3. **Optional Stage 4:** Add Jais model for cross-model validation